In [ ]:
# Intento 2: Experimento, sirvió para decidir. 
# Comparar varios modelos (Gradient Boosting por defecto y regularizado, Random Forest, regresión logística) y tres formas de ordenar los préstamos (por rentabilidad, por ganancia esperada en USD y por probabilidad de pago).
# Cambios respecto del intento anterior: Cambiamos la validación: los préstamos de test son mucho más chicos, así que medimos la ganancia solo sobre préstamos de hasta USD 11.000 y promediamos 5 particiones con distintas semillas.

import pandas as pd
import numpy as np

train = pd.read_csv('data/solicitudes_train.csv')
test  = pd.read_csv('data/solicitudes_test.csv')
banco = pd.read_csv('data/comportamiento_bancario.csv')

In [2]:
# Juntamos train y test para calcular las features igual en los dos
solicitudes = pd.concat([train, test])[['ID_Solicitud', 'ID_Usuario', 'Fecha_Solicitud']]

# Cada solicitud se cruza con todos los meses de su usuario
hist = solicitudes.merge(banco, on='ID_Usuario', how='left')

# Calculamos cuántos meses antes de la solicitud es cada registro de comportamiento bancario
fecha_sol = pd.to_datetime(hist['Fecha_Solicitud'])
fecha_mes = pd.to_datetime(hist['Mes_Referencia']) #mes de cada registro de comportamiento bancario
hist['meses_atras'] = (fecha_sol.dt.year - fecha_mes.dt.year) * 12 + (fecha_sol.dt.month - fecha_mes.dt.month)

# Usamos solo los 12 meses anteriores a la solicitud
VENTANA = 12
hist = hist[(hist['meses_atras'] >= 1) & (hist['meses_atras'] <= VENTANA)]

In [3]:
hist['tuvo_atraso_30'] = hist['Dias_Atraso_Otras_Deudas'] > 30

feat = hist.groupby('ID_Solicitud').agg(
    saldo_prom      = ('Saldo_Promedio_Mensual', 'mean'),
    saldo_min       = ('Saldo_Promedio_Mensual', 'min'),
    saldo_std       = ('Saldo_Promedio_Mensual', 'std'),
    atraso_max      = ('Dias_Atraso_Otras_Deudas', 'max'),
    atraso_prom     = ('Dias_Atraso_Otras_Deudas', 'mean'),
    meses_atraso_30 = ('tuvo_atraso_30', 'sum'),   # cuántos meses debió más de 30 días
    meses_informados= ('Mes_Referencia', 'count'), # cuántos meses de registro tenés
)

# Filtramos por los ultimos 3 meses para evaluar el comportamiento reciente
ult3 = hist[hist['meses_atras'] <= 3]
feat_ult3 = ult3.groupby('ID_Solicitud').agg(
    saldo_prom_ult3 = ('Saldo_Promedio_Mensual', 'mean'),
    atraso_max_ult3 = ('Dias_Atraso_Otras_Deudas', 'max'),
)

# Unimos las features de los ultimos 3 meses con las features generales
feat = feat.join(feat_ult3)
feat['tendencia_saldo'] = feat['saldo_prom_ult3'] / feat['saldo_prom']  # >1 = viene mejorando
feat = feat.reset_index()

In [4]:
# Unimos las features con los datasets de train y test
train_f = train.merge(feat, on='ID_Solicitud', how='left')
test_f  = test.merge(feat,  on='ID_Solicitud', how='left')
print(train_f.shape, test_f.shape)

(32416, 18) (3061, 17)


In [5]:
# Verificamos si a partir de sus sueldo, puede pagar la cuota del préstamo y si su saldo promedio es mayor que su sueldo
for df in (train_f, test_f):
    df['cuota_sobre_ingreso'] = df['Monto_Solicitado'] / df['Ingreso_Mensual']   
    df['saldo_sobre_ingreso'] = df['saldo_prom'] / df['Ingreso_Mensual']

# Motivo_Prestamo es texto: lo pasamos a categoría, con las mismas categorías en train y test
motivos = sorted(train_f['Motivo_Prestamo'].unique())
for df in (train_f, test_f):
    df['Motivo_Prestamo'] = pd.Categorical(df['Motivo_Prestamo'], categories=motivos)

FEATURES = ['Monto_Solicitado', 'Tasa_Interes', 'Ingreso_Mensual', 'Motivo_Prestamo',
            'saldo_prom', 'saldo_min', 'saldo_std', 'atraso_max', 'atraso_prom', 'meses_atraso_30',
            'meses_informados', 'saldo_prom_ult3', 'atraso_max_ult3', 'tendencia_saldo',
            'cuota_sobre_ingreso', 'saldo_sobre_ingreso']

# Datasets de train y test para el modelo 
X      = train_f[FEATURES]
y      = train_f['Estado_Final']
grupos = train_f['ID_Usuario']
X_test = test_f[FEATURES]

PRESUPUESTO = 2_000_000

In [ ]:
# Entrenamiento del modelo de Gradient Boosting regularizado
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

# evaluamos la ganancia real obtenida con nuestra regla de decisión, dado un dataset y las probabilidades de pago estimadas por el modelo
def evaluar(val, p_val, ordenar_por='rentabilidad'):
    """Aplica nuestra regla de decisión y devuelve la ganancia REAL obtenida."""
    val = val.copy()
    val['p_pago'] = p_val
    val['ganancia_esperada'] = (val['p_pago'] * val['Monto_Solicitado'] * val['Tasa_Interes']
                                - (1 - val['p_pago']) * val['Monto_Solicitado'])
    val['rentabilidad'] = val['ganancia_esperada'] / val['Monto_Solicitado']
    val = val.sort_values(ordenar_por, ascending=False)
    val['acumulado'] = val['Monto_Solicitado'].cumsum()
    val['aprobado'] = ((val['ganancia_esperada'] > 0) & (val['acumulado'] <= PRESUPUESTO)).astype(int)
    val['resultado'] = np.where(val['Estado_Final'] == 1,
                                val['Monto_Solicitado'] * val['Tasa_Interes'],
                                -val['Monto_Solicitado'])
    return val.loc[val['aprobado'] == 1, 'resultado'].sum()

In [7]:
# Validación "vieja": una sola partición, con préstamos de todos los tamaños
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
m = HistGradientBoostingClassifier(categorical_features='from_dtype', random_state=42).fit(X_train, y_train)
p = m.predict_proba(X_val)[:, 1]
val = train_f.loc[X_val.index]

print('Orden por rentabilidad:     ', round(evaluar(val, p)))
print('Orden por ganancia en USD:  ', round(evaluar(val, p, 'ganancia_esperada')), ' <- parece mejor, pero es suerte')

Orden por rentabilidad:      478581
Orden por ganancia en USD:   602322  <- parece mejor, pero es suerte


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Función para preparar los datos para Random Forest
def preparar_rf(Z):
    """Random Forest no acepta categorías ni NaN: los convertimos a números."""
    Z = Z.copy()
    Z['Motivo_Prestamo'] = Z['Motivo_Prestamo'].cat.codes
    return Z.fillna(0)

# Función para preparar los datos para regresión logística
def preparar_lr(Z):
    """La regresión logística necesita todo numérico y sin NaN."""
    Z = pd.get_dummies(Z, columns=['Motivo_Prestamo'])   # one-hot: una columna 0/1 por motivo
    return Z.fillna(0)

In [9]:
# Entrenamos varios modelos y comparamos resultados
resultados = {'HGB orden rentabilidad': [], 'HGB orden ganancia': [], 'HGB orden prob. pago': [],
              'HGB regularizado': [], 'Random Forest': [], 'Regresión logística': []}

for semilla in range(5):
    print('Repetición', semilla)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=semilla)
    
    # Validamos solo con préstamos parecidos a los de test (hasta 11.000)
    chicos = X_val['Monto_Solicitado'] <= 11000
    X_val = X_val[chicos]
    val = train_f.loc[X_val.index]

    m = HistGradientBoostingClassifier(categorical_features='from_dtype', random_state=42).fit(X_train, y_train)
    p = m.predict_proba(X_val)[:, 1]
    resultados['HGB orden rentabilidad'].append(evaluar(val, p))
    resultados['HGB orden ganancia'].append(evaluar(val, p, 'ganancia_esperada'))
    resultados['HGB orden prob. pago'].append(evaluar(val, p, 'p_pago'))

    m = HistGradientBoostingClassifier(categorical_features='from_dtype', max_depth=4, min_samples_leaf=50,
                                       learning_rate=0.05, max_iter=400, random_state=42).fit(X_train, y_train)
    resultados['HGB regularizado'].append(evaluar(val, m.predict_proba(X_val)[:, 1]))

    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=42).fit(preparar_rf(X_train), y_train)
    resultados['Random Forest'].append(evaluar(val, m.predict_proba(preparar_rf(X_val))[:, 1]))

    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(preparar_lr(X_train), y_train)
    resultados['Regresión logística'].append(evaluar(val, m.predict_proba(preparar_lr(X_val))[:, 1]))

# Mostramos los resultados promedio
pd.DataFrame(resultados).mean().round().sort_values(ascending=False)

Repetición 0


Repetición 1


Repetición 2


Repetición 3


Repetición 4


HGB regularizado          530909.0
HGB orden rentabilidad    530571.0
Regresión logística       521936.0
Random Forest             516895.0
HGB orden ganancia        481498.0
HGB orden prob. pago      467625.0
dtype: float64

In [ ]:
# Conclusiones
# - Ordenar por rentabilidad (ganancia esperada por dólar) es la decisión que más impacta: con presupuesto fijo conviene elegir lo que más rinde por dólar.
# - Ordenar por ganancia en USD concentra el presupuesto en pocos préstamos grandes (más riesgo, menos ganancia).
# - Ordenar por probabilidad de pago ignora la tasa: elige préstamos muy seguros pero poco rentables.
# - Random Forest queda por debajo de Gradient Boosting.
# - HGB regularizado ≈ HGB por defecto: el modelo no estaba sobreajustando. 
# - Regresión logística queda apenas por debajo.